# Xarray-Spatial I/O: COG Overview Generation

Cloud Optimized GeoTIFFs (COGs) store internal overview images at reduced
resolution so map viewers can fetch the right zoom level without downloading
the full file. xarray-spatial generates these overviews during `to_geotiff()`
with `cog=True`, with no GDAL post-processing required.

### Stable COG contract

The local COG writer and the local COG reader are tagged `stable` in `xrspatial.geotiff.SUPPORTED_FEATURES` (`writer.cog` and `reader.local_cog`). Axis-aligned 2D / 3D rasters, the lossless codecs (`none`, `deflate`, `lzw`, `zstd`, `packbits`), internal overviews, and normal CRS / transform / nodata round-trip are covered by the parity and compliance suites that gate every CI build, so the examples in this notebook all sit inside the stable contract.

A few combinations stay outside the stable contract and keep their existing `advanced` or `experimental` tier:

- The HTTP COG reader (`reader.http_cog`) -- range fetching, redirect handling, and cache behaviour are not contracted yet. Use with care.
- GPU COG read / write.
- Experimental codecs (`lerc`, `jpeg2000` / `j2k`, `lz4`) and the internal-only `jpeg` codec.
- Rotated transforms, external `.tif.ovr` sidecars, file-like destinations with `cog=True`.
- BigTIFF COG (tracked separately).

See the *Stable COG contract* section of the GeoTIFF / COG reference page for the full list.

### What you'll build

1. [Generate synthetic terrain data](#data)
2. [Write a COG with automatic overviews](#write-cog)
3. [Compare resampling methods](#resampling)
4. [Set explicit overview levels](#explicit-levels)
5. [Verify round-trip accuracy](#verify)

Import the libraries we need.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.geotiff import open_geotiff, to_geotiff
from xrspatial.geotiff._header import parse_header, parse_all_ifds

<a id="data"></a>

## Generate synthetic terrain data

A 256x256 elevation surface with Gaussian peaks gives the overviews
something visually meaningful to downsample.

In [ ]:
rng = np.random.RandomState(42)
y = np.linspace(45.0, 44.0, 256)
x = np.linspace(-120.0, -119.0, 256)

# Smooth terrain from Gaussian peaks
xx, yy = np.meshgrid(x, y)
terrain = (
    800 * np.exp(-((xx + 119.5)**2 + (yy - 44.5)**2) / 0.02)
    + 600 * np.exp(-((xx + 119.3)**2 + (yy - 44.7)**2) / 0.05)
    + 100 * rng.rand(256, 256)
).astype(np.float32)

da = xr.DataArray(
    terrain, dims=['y', 'x'],
    coords={'y': y, 'x': x},
    attrs={'crs': 4326},
    name='elevation',
)

The array has two Gaussian peaks plus random noise. Plot it to confirm.

In [ ]:
da.plot.imshow(cmap='terrain', figsize=(6, 5))
plt.title('Synthetic elevation')
plt.tight_layout()
plt.show()

<a id="write-cog"></a>

## Write a COG with automatic overviews

Pass `cog=True` and xarray-spatial halves the dimensions until the smallest
overview fits within a single tile, then writes all levels into the file with
IFDs at the start (the COG layout requirement for efficient HTTP range requests).

In [ ]:
import tempfile, os

tmpdir = tempfile.mkdtemp(prefix='cog_demo_')
cog_path = os.path.join(tmpdir, 'auto_overviews.tif')

to_geotiff(da, cog_path, cog=True, compression='deflate')

# Check how many IFDs (resolution levels) were written
with open(cog_path, 'rb') as f:
    raw = f.read()

header = parse_header(raw)
ifds = parse_all_ifds(raw, header)
for i, ifd in enumerate(ifds):
    label = 'Full resolution' if i == 0 else f'Overview {i}'
    print(f'{label}: {ifd.width} x {ifd.height}')

<a id="resampling"></a>

## Resampling methods

Different data types call for different resampling:
- **mean** (default): continuous data like elevation or temperature
- **nearest**: categorical or index data (land cover classes)
- **mode**: majority-class downsampling for classified rasters
- **min** / **max** / **median**: when you need conservative bounds

In [ ]:
methods = ['mean', 'nearest', 'min', 'max']
fig, axes = plt.subplots(1, len(methods), figsize=(14, 3))

for ax, method in zip(axes, methods):
    path = os.path.join(tmpdir, f'cog_{method}.tif')
    to_geotiff(da, path, cog=True, compression='deflate',
               overview_resampling=method)

    # Read back the first overview level
    result = open_geotiff(path, overview_level=1)
    result.plot.imshow(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(f'{method} ({result.shape[0]}x{result.shape[1]})')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Overview level 1 by resampling method', y=1.02)
plt.tight_layout()
plt.show()

<a id="explicit-levels"></a>

## Explicit overview levels

For fine-grained control, pass `overview_levels` as a list of
decimation factors.

In [ ]:
explicit_path = os.path.join(tmpdir, 'explicit_levels.tif')
to_geotiff(da, explicit_path, cog=True, compression='deflate',
           overview_levels=[2, 4, 8])

with open(explicit_path, 'rb') as f:
    raw = f.read()

header = parse_header(raw)
ifds = parse_all_ifds(raw, header)
for i, ifd in enumerate(ifds):
    label = 'Full resolution' if i == 0 else f'Overview {i}'
    print(f'{label}: {ifd.width} x {ifd.height}')

<a id="verify"></a>

## Verify round-trip

Full-resolution pixel values are preserved through the COG write.

In [ ]:
result = open_geotiff(cog_path)
max_diff = float(np.max(np.abs(result.values - terrain)))
print(f'Max pixel difference: {max_diff}')
assert max_diff < 1e-5, 'Values should round-trip exactly'

In [ ]:
# Clean up
import shutil
shutil.rmtree(tmpdir)

### References

- [COG specification](https://www.cogeo.org/)
- [xarray-spatial `to_geotiff` API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/io.html)
- [TIFF/GeoTIFF overview structure (OGC)](https://docs.ogc.org/is/19-008r4/19-008r4.html)
- [GDAL COG driver](https://gdal.org/drivers/raster/cog.html)